# DummyCell — synthetic solar cell demo

This notebook shows all six simulation modes available in `DummyCell`.
Each mode is useful for testing a different aspect of the software without real hardware.

**Sign convention:** photocurrent is *positive* in the power-generating quadrant (V > 0, J > 0).

In [ ]:
import sys, pathlib
# Make sure the package root is on the path when running from docs/notebooks/
sys.path.insert(0, str(pathlib.Path().resolve().parents[1]))

import numpy as np
import matplotlib.pyplot as plt
from app.simulation.dummy_cell import DummyCell

V_SCAN = np.linspace(1.35, -0.05, 300)  # reverse sweep, V
ILLUMINATION = 100.0  # mW/cm²

## Mode 1 — `ideal_stable`

Textbook single-diode cell with fixed parameters.
Use this mode when you want a clean, noise-free reference curve to verify
that parameter extraction (Voc, Jsc, FF, PCE) works correctly.

In [ ]:
cell = DummyCell(mode="ideal_stable")
J = cell.scan(V_SCAN)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(V_SCAN, J, color="steelblue")
ax.axhline(0, color="k", lw=0.7)
ax.set_xlabel("Voltage (V)"); ax.set_ylabel("J (mA/cm²)")
ax.set_title("ideal_stable")
ax.set_ylim(-5, 25)
plt.tight_layout(); plt.show()
print(f"Jsc ≈ {cell.current_density(0.0):.2f} mA/cm²")

## Mode 2 — `with_hysteresis`

Forward and reverse scans give different efficiencies.
The `hi_fraction` parameter sets the approximate hysteresis index:
`hi_fraction=0.05` → ~5 % HI (planar cells), `hi_fraction=0.20` → ~18 % HI (mesoporous cells).

Use this mode to test that your analysis code correctly flags hysteretic devices.

In [ ]:
for hi, label, color in [
    (0.05,  "hi_fraction=0.05 (planar)",     "steelblue"),
    (0.20,  "hi_fraction=0.20 (mesoporous)", "darkorange"),
]:
    cell = DummyCell(mode="with_hysteresis", hi_fraction=hi)
    J_rev = cell.scan(V_SCAN, scan_direction="reverse")
    J_fwd = cell.scan(V_SCAN, scan_direction="forward")

    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.plot(V_SCAN, J_rev, color=color,        label="reverse")
    ax.plot(V_SCAN, J_fwd, color=color, ls="--", label="forward")
    ax.axhline(0, color="k", lw=0.7)
    ax.set_xlabel("Voltage (V)"); ax.set_ylabel("J (mA/cm²)")
    ax.set_title(f"with_hysteresis — {label}")
    ax.set_ylim(-5, 25); ax.legend()
    plt.tight_layout(); plt.show()

## Mode 3 — `exponential_decay`

Efficiency decays as exp(−t/τ).  Default τ = 100 h.
Use this mode to test ISOS stability tracking: the software should record
a monotonically decreasing PCE_norm over time.

In [ ]:
tau_h = 100.0
cell = DummyCell(mode="exponential_decay", tau_h=tau_h)

times_h = [0, 25, 50, 100, 200]
fig, ax = plt.subplots(figsize=(5, 3.5))
for t_h in times_h:
    cell.set_time_offset(t_h * 3600)
    J = cell.scan(V_SCAN)
    ax.plot(V_SCAN, J, label=f"t = {t_h} h")

ax.axhline(0, color="k", lw=0.7)
ax.set_xlabel("Voltage (V)"); ax.set_ylabel("J (mA/cm²)")
ax.set_title(f"exponential_decay  (τ = {tau_h} h)")
ax.set_ylim(-5, 25); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Mode 4 — `burn_in`

Efficiency starts at 80 % of nominal and rises to 100 % in ~30 min — typical for
fresh perovskite devices that need a light-soaking activation phase.
Use this mode to test that ISOS logging captures the initial rise correctly.

In [ ]:
cell = DummyCell(mode="burn_in")

times_min = [0, 10, 30, 60, 120]
fig, ax = plt.subplots(figsize=(5, 3.5))
for t_min in times_min:
    cell.set_time_offset(t_min * 60)
    J = cell.scan(V_SCAN)
    ax.plot(V_SCAN, J, label=f"t = {t_min} min")

ax.axhline(0, color="k", lw=0.7)
ax.set_xlabel("Voltage (V)"); ax.set_ylabel("J (mA/cm²)")
ax.set_title("burn_in  (rises from 80 % to 100 % in ~30 min)")
ax.set_ylim(-2, 25); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Mode 5 — `noisy`

Ideal parameters plus Gaussian noise on every current reading.
Use this mode to test that parameter extraction is robust to instrument noise,
and to validate that averaging / smoothing logic works.

In [ ]:
rng = np.random.default_rng(42)
cell_ideal = DummyCell(mode="ideal_stable")
cell_noisy = DummyCell(mode="noisy", noise_std=0.3, rng=rng)

J_ideal = cell_ideal.scan(V_SCAN)
J_noisy = cell_noisy.scan(V_SCAN)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(V_SCAN, J_ideal, color="steelblue",    lw=2,   label="ideal")
ax.plot(V_SCAN, J_noisy, color="darkorange",   lw=0.8, label="noisy (σ=0.3 mA/cm²)")
ax.axhline(0, color="k", lw=0.7)
ax.set_xlabel("Voltage (V)"); ax.set_ylabel("J (mA/cm²)")
ax.set_title("noisy")
ax.set_ylim(-5, 25); ax.legend()
plt.tight_layout(); plt.show()

## Mode 6 — `catastrophic_failure`

The cell is ideal until `t_fail_h` hours, then the shunt resistance collapses
by a factor of 1000 (simulating a short circuit defect or delamination).
Use this mode to verify that the software detects and flags sudden PCE drops.

In [ ]:
t_fail_h = 50.0
cell = DummyCell(mode="catastrophic_failure", t_fail_h=t_fail_h)

fig, ax = plt.subplots(figsize=(5, 3.5))
for t_h, label, color in [
    (0,       "t = 0 h (healthy)",   "steelblue"),
    (t_fail_h, f"t = {t_fail_h:.0f} h (failed)", "crimson"),
]:
    cell.set_time_offset(t_h * 3600)
    J = cell.scan(V_SCAN)
    ax.plot(V_SCAN, J, color=color, label=label)

ax.axhline(0, color="k", lw=0.7)
ax.set_xlabel("Voltage (V)"); ax.set_ylabel("J (mA/cm²)")
ax.set_title(f"catastrophic_failure  (fails at t = {t_fail_h:.0f} h)")
ax.set_ylim(-5, 25); ax.legend()
plt.tight_layout(); plt.show()

## Smoke-test: factory round-trip

Demonstrates the `create_sourcemeter("dummy")` entry point used in production code.

In [ ]:
from app.instrument import create_sourcemeter

smu = create_sourcemeter("dummy", sleep_scale=0.0)
smu.connect()
smu.configure(nplc=1, delay_s=0.0, four_wire=False)
smu.set_voltage(0.5)
j = smu.read_current_density(cell_area=0.1)
print(f"J at V=0.5 V: {j:.4f} mA/cm²   (expected ~{DummyCell().current_density(0.5):.4f} mA/cm²)")